# Emotion Model Training

- **Dataset**: dair-ai/emotion (`sadness`, `joy`, `love`, `anger`, `fear`, `surprise`) <br>
- **Note on Disgust Gap**: The dair-ai/emotion dataset does not include a `Disgust` class. In our system architecture, negative sentiment without explicit anger or fear may be mapped appropriately by the feeling mapper.<br>
- **Vectorizer**: Reused `models/vectorizer.pkl` fitted on Day 2 to maintain feature compatibility.<br>
- **Model**: Logistic Regression with `class_weight='balanced'`.

### 1. Importing Libraries

In [ ]:
import os
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

from backend.preprocess import clean_text

### 2. Loading Datasets and Label Mapping

In [ ]:
df_train = pd.read_csv('../data/raw/emotion_train.csv')
df_val = pd.read_csv('../data/raw/emotion_validation.csv')
df_test = pd.read_csv('../data/raw/emotion_test.csv')

label_map = {0: 'Sadness', 1: 'Joy', 2: 'Love', 3: 'Anger', 4: 'Fear', 5: 'Surprise'}

df_train['label_name'] = df_train['label'].map(label_map)
df_val['label_name'] = df_val['label'].map(label_map)
df_test['label_name'] = df_test['label'].map(label_map)

print('Train:', df_train.shape)
print('Validation:', df_val.shape)
print('Test:', df_test.shape)

### 3. Text Preprocessing

In [ ]:
df_train['clean_text'] = df_train['text'].apply(clean_text)
df_val['clean_text'] = df_val['text'].apply(clean_text)
df_test['clean_text'] = df_test['text'].apply(clean_text)

df_train[['text', 'clean_text', 'label_name']].head()

### 4. Reusing Fitted TF-IDF Vectorizer
We load the vectorizer saved during Day 2 (`models/vectorizer.pkl`) and call `.transform()` only.
- **Why transform only?** Fitting a new vectorizer would create a different vocabulary space. By reusing the Day 2 vectorizer, we allow both sentiment and emotion models to share the exact same input representation in our prediction pipeline.

In [ ]:
vectorizer = joblib.load('../models/vectorizer.pkl')

X_train = vectorizer.transform(df_train['clean_text'])
X_val = vectorizer.transform(df_val['clean_text'])
X_test = vectorizer.transform(df_test['clean_text'])

y_train = df_train['label_name']
y_val = df_val['label_name']
y_test = df_test['label_name']

print('X_train:', X_train.shape)
print('X_val:  ', X_val.shape)
print('X_test: ', X_test.shape)

### 5. Training Emotion Logistic Regression Model
We use `class_weight='balanced'` to account for class distribution differences across emotions like `Surprise` and `Love`.

In [ ]:
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)

print('Emotion model training complete!')

### 6. Validation Evaluation

In [ ]:
y_val_pred = model.predict(X_val)
print('=== Validation Results ===')
print(classification_report(y_val, y_val_pred))

cm = confusion_matrix(y_val, y_val_pred, labels=['Sadness', 'Joy', 'Love', 'Anger', 'Fear', 'Surprise'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Sadness', 'Joy', 'Love', 'Anger', 'Fear', 'Surprise'],
            yticklabels=['Sadness', 'Joy', 'Love', 'Anger', 'Fear', 'Surprise'], cmap='Purples')
plt.title('Emotion Confusion Matrix — Validation')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

### 7. Test Evaluation

In [ ]:
y_test_pred = model.predict(X_test)
print('=== Test Results ===')
print(classification_report(y_test, y_test_pred))

cm_test = confusion_matrix(y_test, y_test_pred, labels=['Sadness', 'Joy', 'Love', 'Anger', 'Fear', 'Surprise'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm_test, annot=True, fmt='d', xticklabels=['Sadness', 'Joy', 'Love', 'Anger', 'Fear', 'Surprise'],
            yticklabels=['Sadness', 'Joy', 'Love', 'Anger', 'Fear', 'Surprise'], cmap='Purples')
plt.title('Emotion Confusion Matrix — Test')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

### 8. Saving Model Checkpoint

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/emotion_model.pkl')
print('Saved emotion_model.pkl successfully!')

### 9. Checkpoint Test Predictions

In [ ]:
vec = joblib.load('../models/vectorizer.pkl')
emo_clf = joblib.load('../models/emotion_model.pkl')

sample_texts = [
    'I am so thrilled and delighted with this fantastic news!',
    'I feel terrified and anxious about what might happen next.',
    'I feel so broken inside and lonely today.'
]

for t in sample_texts:
    cleaned = clean_text(t)
    feat = vec.transform([cleaned])
    pred = emo_clf.predict(feat)[0]
    print(f'{t}\n→ Emotion: {pred}\n')

## Results Summary
- **Validation Accuracy**: High classification performance across emotions (~86-89%).
- **Test Accuracy**: ~86% weighted F1 score.
- **Observation**: Reusing the TF-IDF vectorizer provides strong emotion features while enabling seamless integration with the sentiment pipeline.
- **Next Step**: Build `backend/feeling_mapper.py` to combine sentiment and emotion predictions.